In [2]:
import yfinance as yf
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

TICKERS = [
    "LLY", "NVO", "SNY", "VRTX", "OTSKY",
    "CRSP", "SANA", "EVO", "LCTX", "HUMA",
    "GNPX", "IPSC", "SABS", "SEOVF", "NCEL",
    "FLUI.ST", "NXTCL.ST", "IMCR", "CELZ", "ELDN",
    "ADOC.PA"
]

SPX_TICKER    = "^GSPC"
SECTOR_TICKER = "IBB"

# ── Set the "as of" date you want ────────────────────────────────────────────
AS_OF = pd.Timestamp("2026-07-06", tz="America/New_York")
END   = (AS_OF + pd.Timedelta(days=1)).strftime("%Y-%m-%d")


# ── Weekly return: close 5 trading days ago → latest close ──────────────────
def get_weekly_return(close: pd.Series):
    """
    True 5-trading-day return: price[t] / price[t-5] - 1.
    Takes a Close price Series (already converted to USD for foreign tickers).
    Requires at least 6 rows (today + 5 prior days).
    """
    if close is None or close.empty or len(close) < 6:
        return None
    last6 = close.tail(6)
    return float(last6.iloc[-1] / last6.iloc[0] - 1)


# ── YTD return: last close of prior year → latest close ─────────────────────
def get_ytd_return(close: pd.Series, as_of: pd.Timestamp):
    """
    Uses the last trading day of the prior calendar year as the base,
    matching how Bloomberg / most data providers calculate YTD.
    Takes a Close price Series (already converted to USD for foreign tickers).
    """
    if close is None or close.empty:
        return None
    year = as_of.year
    prior_year = close[close.index.year == year - 1]
    current_year = close[close.index.year == year]
    if prior_year.empty or current_year.empty:
        return None
    base   = float(prior_year.iloc[-1])
    latest = float(current_year.iloc[-1])
    return latest / base - 1


# ── Market cap classification ────────────────────────────────────────────────
def classify_market_cap(market_cap_usd: float):
    if market_cap_usd is None:
        return None
    if market_cap_usd >= 10e9:
        return "Large-cap"
    elif market_cap_usd >= 2e9:
        return "Mid-cap"
    elif market_cap_usd >= 300e6:
        return "Small-cap"
    else:
        return "Micro-cap"


# ── History fetch capped at AS_OF ────────────────────────────────────────────
def history_as_of(ticker: yf.Ticker, lookback_days: int = 420):
    """
    Fetch ~420 calendar days of daily history and trim to AS_OF.
    420 days covers 252+ trading days comfortably for the 52-week range.
    """
    start = (AS_OF - pd.Timedelta(days=lookback_days)).strftime("%Y-%m-%d")
    try:
        hist = ticker.history(start=start, end=END, interval="1d")
    except Exception as e:
        print(f"  [WARN] history fetch failed: {e}")
        return pd.DataFrame()
    if hist is None or hist.empty:
        return pd.DataFrame()
    return hist[hist.index <= AS_OF]


# ── FX helpers ───────────────────────────────────────────────────────────────
_FX_HIST_CACHE: dict = {}

def get_fx_history_to_usd(ccy: str) -> pd.Series | None:
    """
    Daily FX close series (1 unit of local ccy → USD), trimmed to AS_OF.
    Returns None for USD (meaning "no conversion needed") and also None
    if the FX history can't be fetched (callers must handle that case).
    Fetches the full 420-day lookback so historical prices (YTD base,
    weekly chart) can be converted at the FX rate of their own date.
    """
    ccy = (ccy or "").upper()
    if ccy in ("USD", ""):
        return None
    if ccy in _FX_HIST_CACHE:
        return _FX_HIST_CACHE[ccy]
    fx_tkr = f"{ccy}USD=X"
    fx_hist = history_as_of(yf.Ticker(fx_tkr), lookback_days=420)
    if fx_hist is None or fx_hist.empty:
        print(f"  [WARN] FX history not found for {ccy}, will be None")
        _FX_HIST_CACHE[ccy] = None
        return None
    s = fx_hist["Close"].astype(float)
    _FX_HIST_CACHE[ccy] = s
    return s

def fx_aligned_to_index(index: pd.DatetimeIndex, ccy: str) -> pd.Series | None:
    """
    FX rates (local ccy → USD) aligned to a stock's trading dates.
    FX and stock calendars differ (different exchanges/holidays), so rates
    are aligned by calendar date and forward-filled across gaps.
    Returns a series of 1.0s for USD tickers, or None if the ticker is
    non-USD and no FX history is available.
    """
    ccy = (ccy or "").upper()
    if ccy in ("USD", ""):
        return pd.Series(1.0, index=index)
    fx_series = get_fx_history_to_usd(ccy)
    if fx_series is None:
        return None
    # Align by calendar date (indexes carry different timezones/timestamps)
    fx_by_date = pd.Series(fx_series.values, index=pd.Index(fx_series.index.date))
    fx_by_date = fx_by_date[~fx_by_date.index.duplicated(keep="last")].sort_index()
    stock_dates = pd.Index(index.date)
    union_idx = fx_by_date.index.union(stock_dates)
    fx_aligned = fx_by_date.reindex(union_idx).ffill().bfill().reindex(stock_dates)
    return pd.Series(fx_aligned.values, index=index)

def series_to_usd(hist: pd.DataFrame, column: str, ccy: str) -> pd.Series | None:
    """
    Convert one daily price column (Close, Low, High, ...) to USD using
    the FX rate of each date. Returns None if hist is empty or the FX
    history for a non-USD ticker is unavailable.
    """
    if hist is None or hist.empty:
        return None
    fx = fx_aligned_to_index(hist.index, ccy)
    if fx is None:
        return None
    return hist[column].astype(float) * fx

def to_usd(value, fx) -> float | None:
    if value is None or fx is None:
        return None
    return float(value) * float(fx)

# ── Weekly closing history for chart (USD, last 52 weeks) ────────────────────
def get_weekly_history_usd(close_usd: pd.Series) -> list:
    """
    Resample a daily USD Close series to weekly (Friday close) over the last
    52 weeks and return as list of [timestamp_ms, price] pairs.
    Gaps (e.g. missing weeks for illiquid stocks) are kept as-is.
    """
    if close_usd is None or close_usd.empty:
        return []
    hist_52w = close_usd.tail(365)   # calendar days — plenty of buffer
    if hist_52w.empty:
        return []
    weekly = hist_52w.resample("W-FRI").last().dropna()
    result = []
    for ts, price in weekly.items():
        ts_ms = int(ts.timestamp() * 1000)
        result.append([ts_ms, round(float(price), 4)])
    return result



# ── Benchmarks ───────────────────────────────────────────────────────────────
print("Fetching benchmarks...")
spx_hist = history_as_of(yf.Ticker(SPX_TICKER))
ibb_hist = history_as_of(yf.Ticker(SECTOR_TICKER))
spx_week_ret = get_weekly_return(spx_hist["Close"] if not spx_hist.empty else None) or 0.0
ibb_week_ret = get_weekly_return(ibb_hist["Close"] if not ibb_hist.empty else None) or 0.0
print(f"  S&P 500 weekly: {spx_week_ret*100:.2f}%  |  IBB weekly: {ibb_week_ret*100:.2f}%")

# Capture IBB weekly history for the dashboard chart (already USD)
ibb_weekly_history = get_weekly_history_usd(ibb_hist["Close"] if not ibb_hist.empty else None)

# Capture S&P 500 weekly history for record keeping (not used by the dashboard)
spx_weekly_history = get_weekly_history_usd(spx_hist["Close"] if not spx_hist.empty else None)


# ── Per-ticker metrics ───────────────────────────────────────────────────────
def get_metrics(ticker: str) -> dict:
    print(f"  {ticker}...")
    t = yf.Ticker(ticker)

    try:
        info = t.info or {}
    except Exception as e:
        print(f"  [WARN] {ticker}: info fetch failed ({e})")
        info = {}

    local_ccy = info.get("currency")

    # Spot FX (as-of) — only needed for market cap, which is a current
    # snapshot from yfinance and has no historical series to convert
    fx_series = get_fx_history_to_usd(local_ccy)
    if (local_ccy or "").upper() in ("USD", ""):
        fx_to_usd = 1.0
    elif fx_series is not None:
        fx_to_usd = float(fx_series.iloc[-1])
    else:
        fx_to_usd = None

    hist = history_as_of(t)

    # Full daily price series in USD (each date converted at that date's FX rate)
    close_usd = series_to_usd(hist, "Close", local_ccy)
    low_usd   = series_to_usd(hist, "Low",   local_ccy)
    high_usd  = series_to_usd(hist, "High",  local_ccy)

    if not hist.empty and close_usd is not None:
        # Latest close in USD (converted at its own date's FX rate)
        close_price_usd = float(close_usd.iloc[-1])
        # 52-week range as it actually was in USD: min/max of the
        # per-date-converted series, not local extremes at today's FX
        low_52w_usd  = float(low_usd.tail(252).min())
        high_52w_usd = float(high_usd.tail(252).max())
        # Returns computed on the USD series so FX moves are included,
        # making them directly comparable to the S&P 500 / IBB benchmarks
        week_ret = get_weekly_return(close_usd)
        ytd_ret  = get_ytd_return(close_usd, AS_OF)

        # Volume — warn if looks unreliable (common with non-US OTC tickers)
        vol_series = hist["Volume"].replace(0, pd.NA)
        vol_100d   = vol_series.tail(100).mean()
        vol_week   = vol_series.tail(5).mean()
        rel_vol    = (vol_week / vol_100d) if (pd.notna(vol_100d) and vol_100d > 0) else None

        if ticker in ("FLUI.ST", "NXTCL.ST") and (rel_vol is None or vol_100d < 100):
            print(f"  [WARN] {ticker}: volume data may be unreliable (vol_100d={vol_100d})")
    else:
        if hist.empty:
            print(f"  [WARN] {ticker}: no price history returned")
        else:
            print(f"  [WARN] {ticker}: FX history unavailable, USD metrics skipped")
        close_price_usd = low_52w_usd = high_52w_usd = None
        week_ret = ytd_ret = None
        vol_100d = vol_week = rel_vol = None

    if week_ret is not None:
        excess_spx = (week_ret - spx_week_ret) * 100
        excess_ibb = (week_ret - ibb_week_ret) * 100
    else:
        excess_spx = excess_ibb = None

    market_cap_local = info.get("marketCap")
    market_cap_usd   = to_usd(market_cap_local, fx_to_usd)

    # Weekly history for chart (historical FX applied per-date)
    weekly_hist_usd = get_weekly_history_usd(close_usd)

    return {
        "As Of":                                   AS_OF.strftime("%Y-%m-%d"),
        "Ticker":                                  ticker,
        "Company Name":                            info.get("shortName"),
        "Industry Group":                          info.get("industry"),
        "Local Currency":                          local_ccy,
        "FX to USD (as-of)":                       fx_to_usd,
        "Market Cap (USD)":                        market_cap_usd,
        "Market Cap Group":                        classify_market_cap(market_cap_usd),
        "Closing Price (USD)":                     close_price_usd,
        "52 Week Low (USD)":                       low_52w_usd,
        "52 Week High (USD)":                      high_52w_usd,
        "1-Week Return (Past 5 Trading Days)":     week_ret,
        "Excess Weekly Return vs. S&P 500 (ppt)":  excess_spx,
        "Excess Weekly Return vs. IBB (ppt)":      excess_ibb,
        "Total Return (YTD)":                      ytd_ret,
        "Average Daily Volume (100D)":             vol_100d,
        "Avg Daily Volume (Last 5 Days)":          vol_week,
        "Relative Volume (Last 5D vs 100D)":       rel_vol,
        "EPS (Basic)":                             info.get("trailingEps"),
        "P/E (Trailing)":                          info.get("trailingPE"),
        "Current Ratio":                           info.get("currentRatio"),
        "Weekly History (USD)":                    weekly_hist_usd,
    }


# ── Run ──────────────────────────────────────────────────────────────────────
print("\nFetching portfolio metrics...")
rows = []
for tkr in TICKERS:
    rows.append(get_metrics(tkr))

df = pd.DataFrame(rows)

# Sanity check: flag any tickers with missing price or return data
missing = df[df["Closing Price (USD)"].isna()]["Ticker"].tolist()
if missing:
    print(f"\n[WARN] No price data for: {missing}")

no_ytd = df[df["Total Return (YTD)"].isna()]["Ticker"].tolist()
if no_ytd:
    print(f"[WARN] No YTD return for: {no_ytd}")

outfile = f"jdca_weekly_metrics_asof_{AS_OF.strftime('%Y-%m-%d')}.xlsx"
df.to_excel(outfile, index=False)
print(f"\nDone. Saved to {outfile}")
print(df[["Ticker","Closing Price (USD)","1-Week Return (Past 5 Trading Days)","Total Return (YTD)"]].to_string(index=False))

# Also save IBB history as a top-level attribute for the xlsx
# (stored separately since it's the same for all tickers)
import json as _json
ibb_hist_json = _json.dumps(ibb_weekly_history)
print(f"\nIBB weekly history: {len(ibb_weekly_history)} data points")
print(f"First: {ibb_weekly_history[0] if ibb_weekly_history else 'none'}  Last: {ibb_weekly_history[-1] if ibb_weekly_history else 'none'}")

# Save IBB history to a sidecar JSON file alongside the xlsx
ibb_out = f"jdca_ibb_history_asof_{AS_OF.strftime('%Y-%m-%d')}.json"
with open(ibb_out, "w") as _f:
    _json.dump(ibb_weekly_history, _f)
print(f"IBB history saved to {ibb_out}")

# Save S&P 500 history to a sidecar JSON file (record keeping only —
# not consumed by the site dashboard)
print(f"\nSPX weekly history: {len(spx_weekly_history)} data points")
print(f"First: {spx_weekly_history[0] if spx_weekly_history else 'none'}  Last: {spx_weekly_history[-1] if spx_weekly_history else 'none'}")
spx_out = f"jdca_spx_history_asof_{AS_OF.strftime('%Y-%m-%d')}.json"
with open(spx_out, "w") as _f:
    _json.dump(spx_weekly_history, _f)
print(f"SPX history saved to {spx_out}")


Fetching benchmarks...
  S&P 500 weekly: 2.49%  |  IBB weekly: 4.24%

Fetching portfolio metrics...
  LLY...
  NVO...
  SNY...
  VRTX...
  OTSKY...
  CRSP...
  SANA...
  EVO...
  LCTX...
  HUMA...
  GNPX...
  IPSC...
  SABS...
  SEOVF...
  NCEL...
  FLUI.ST...
  NXTCL.ST...
  IMCR...
  CELZ...
  ELDN...
  ADOC.PA...

Done. Saved to jdca_weekly_metrics_asof_2026-07-06.xlsx
  Ticker  Closing Price (USD)  1-Week Return (Past 5 Trading Days)  Total Return (YTD)
     LLY          1200.060059                            -0.006671            0.120456
     NVO            49.259998                             0.024756            0.003667
     SNY            42.610001                            -0.007916           -0.071939
    VRTX           529.590027                             0.077848            0.168145
   OTSKY            36.200001                             0.067532            0.277797
    CRSP            61.889999                             0.126912            0.180206
    SANA        